# Federated Learning Classification

This notebook demonstrates privacy-preserving classification using Federated Learning.

We'll use:
- **Random Forest Classifier** for ensemble learning
- **Logistic Regression** for linear classification
- **Gradient Boosting** for boosted trees

All training happens locally on each party's data with secure aggregation.

In [ ]:
import numpy as np
import random
import time

import secretflow as sf
import secretflow.distributed as sfd
from secretflow.data import FedNdarray, PartitionWay
from secretflow.distributed.const import DISTRIBUTION_MODE

# Initialize SecretFlow
base_port = random.randint(10000, 60000)
cluster_config = {
    'parties': {
        'alice': {'address': f'localhost:{base_port}', 'listen_addr': f'0.0.0.0:{base_port}'},
        'bob': {'address': f'localhost:{base_port+1}', 'listen_addr': f'0.0.0.0:{base_port+1}'},
    },
    'self_party': 'alice'
}

sfd.init(DISTRIBUTION_MODE.PRODUCTION, cluster_config=cluster_config)

alice = sf.PYU('alice')
bob = sf.PYU('bob')
devices = {"alice": alice, "bob": bob}

print("✓ SecretFlow initialized")

## Create Classification Dataset

We'll create a synthetic binary classification dataset with clear class separation.

In [ ]:
np.random.seed(42)
n_samples = 800
n_features = 10

# Create two clusters
X_class0 = np.random.randn(n_samples // 2, n_features) - 1
X_class1 = np.random.randn(n_samples // 2, n_features) + 1
X = np.vstack([X_class0, X_class1]).astype(np.float32)
y = np.array([0] * (n_samples // 2) + [1] * (n_samples // 2)).astype(np.int32)

# Shuffle
indices = np.random.permutation(n_samples)
X, y = X[indices], y[indices]

# Vertical partitioning
X_alice = X[:, :5]
X_bob = X[:, 5:]

# Create federated data
fed_X = FedNdarray(
    partitions={
        alice: alice(lambda x: x)(X_alice),
        bob: bob(lambda x: x)(X_bob),
    },
    partition_way=PartitionWay.VERTICAL
)

fed_y = FedNdarray(
    partitions={alice: alice(lambda x: x)(y)},
    partition_way=PartitionWay.HORIZONTAL
)

print(f"✓ Data: {n_samples} samples, {n_features} features")
print(f"  Classes: {np.bincount(y)}")

## Train Multiple Classifiers

Let's compare different FL classifiers on our dataset.

In [ ]:
from secretlearn.federated_learning.linear_models.logistic_regression import FLLogisticRegression
from secretlearn.federated_learning.ensemble.random_forest_classifier import FLRandomForestClassifier
from secretlearn.federated_learning.ensemble.gradient_boosting_classifier import FLGradientBoostingClassifier

results = {}

# 1. Logistic Regression
print("Training FLLogisticRegression...")
start = time.time()
lr_model = FLLogisticRegression(devices)
lr_model.fit(fed_X, fed_y)
results['LogisticRegression'] = time.time() - start
print(f"  ✓ {results['LogisticRegression']*1000:.2f}ms")

# 2. Random Forest
print("Training FLRandomForestClassifier...")
start = time.time()
rf_model = FLRandomForestClassifier(devices)
rf_model.fit(fed_X, fed_y)
results['RandomForest'] = time.time() - start
print(f"  ✓ {results['RandomForest']*1000:.2f}ms")

# 3. Gradient Boosting
print("Training FLGradientBoostingClassifier...")
start = time.time()
gb_model = FLGradientBoostingClassifier(devices)
gb_model.fit(fed_X, fed_y)
results['GradientBoosting'] = time.time() - start
print(f"  ✓ {results['GradientBoosting']*1000:.2f}ms")

print("\n✓ All classifiers trained successfully!")